# **Introduction to PyTorch Optimizers and `nn.Linear`**

This notebook demonstrates how to move from manual parameter updates to using PyTorch's built-in optimization algorithms, followed by an introduction to creating and training linear models using the `torch.nn.Linear` module.

We will use the previous example from our autograd and backpropagation notebook ($y = 2x$). Starting with an initial weight of $0$, the target weight is $2$, and our goal is to minimize the error to $0$.

In [5]:
# previously our model was y = wx
import torch

x = torch.tensor([1.0, 2.0, 3.0])
y_true = torch.tensor([2.0, 4.0, 6.0])

weight = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.01

for epoch in range(20):
  # Forward pass: make predictions
  y_pred = weight * x

  # calculate loss
  loss = ((y_pred - y_true) ** 2).mean()

  # clear gradients from the previous iteration
  if weight.grad is not None:
    weight.grad.zero_()

  # backward pass: calculate the gradient
  loss.backward()

  # update the weight
  with torch.no_grad():
    weight -= learning_rate * weight.grad

  if epoch % 2 == 0:
      print(
          f"Epoch {epoch:2d} | "
          f"Loss: {loss.item():.4f} | "
          f"Weight: {weight.item():.4f}"
      )

print("\nFinal Weight:", weight.item())

Epoch  0 | Loss: 18.6667 | Weight: 0.1867
Epoch  2 | Loss: 12.6141 | Weight: 0.5094
Epoch  4 | Loss: 8.5241 | Weight: 0.7746
Epoch  6 | Loss: 5.7602 | Weight: 0.9927
Epoch  8 | Loss: 3.8925 | Weight: 1.1719
Epoch 10 | Loss: 2.6304 | Weight: 1.3193
Epoch 12 | Loss: 1.7775 | Weight: 1.4404
Epoch 14 | Loss: 1.2012 | Weight: 1.5400
Epoch 16 | Loss: 0.8117 | Weight: 1.6219
Epoch 18 | Loss: 0.5485 | Weight: 1.6892

Final Weight: 1.7181727886199951


#### **Note on Hyperparameters**
By increasing the number of epochs or adjusting the learning rate, we can achieve a training loss closer to zero. Playing with these hyperparameters allows us to observe how optimization patterns change.

As you can see from the outputs above, the loss steadily decreases and the weight converges toward its optimal value of `2.0`.

## **Optimizers and `torch.nn.Linear`**

In this section, we will cover three key concepts:
1. **What an optimizer does** in PyTorch.
2. **How `nn.Linear` represents** a trainable mathematical function (layer).
3. **How to build and train** your first small neural network model using the standard PyTorch workflow.

In [ ]:
# From manual gradient descent to an optimizer
#e.g

# with torch.no_grad():
#   weight -= learning_rate * weight.grad

# this will work but for millions of parameters it's not a good choice.
# pytorch have optimizers for this.

### **Why use an Optimizer?**

Instead of updating model weights and biases manually (e.g., using `with torch.no_grad():`), PyTorch provides automated optimizers.

```
Forward Pass ──> Calculate Loss ──> Backward Pass (Gradients) ──> Optimizer (Update Parameters)
```

**Commonly used optimizers:**
* **`torch.optim.SGD`**: Stochastic Gradient Descent. It directly applies the standard gradient descent formula.
* **`torch.optim.Adam`**: An adaptive optimizer widely used in deep learning for faster convergence.

We will start with **SGD** because it directly mirrors the manual updates we just performed.

In [6]:
#my first optimizer
import torch

weight = torch.tensor(0.0, requires_grad=True)

optimizer = torch.optim.SGD(
    [weight],
    lr=0.01
)

print(weight)

tensor(0., requires_grad=True)


In [8]:
# Real example
x = torch.tensor([1.0, 2.0, 3.0])
y_true = torch.tensor([2.0, 4.0, 6.0])

weight = torch.tensor(0.0, requires_grad=True)

optimizer = torch.optim.SGD([weight], lr=0.01)

for epoch in range(20):
  y_pred = weight * x

  loss = ((y_pred - y_true) ** 2).mean()

  # clear old gradients
  optimizer.zero_grad()

  loss.backward()

  # update parameters
  optimizer.step()

print("Learned weight:", weight.item())
print("Final loss:", loss.item())

Learned weight: 1.7181727886199951
Final loss: 0.4508970081806183


### **The Optimization Loop Sequence**

Always remember the exact order of operations inside your training loop:

| Code Step | Purpose |
| :--- | :--- |
| `optimizer.zero_grad()` | Clear out accumulated gradients from the previous iteration. |
| `loss.backward()` | Compute the gradients of the loss with respect to the parameters using Autograd. |
| `optimizer.step()` | Update the parameter values using the calculated gradients. |

> **Important:** The optimizer does *not* calculate the gradients itself; Autograd calculates them during `loss.backward()`, and the optimizer simply applies them to update the weights.

---

## **Introducing `nn.Linear`**

Previously, we used a manual equation: $y = w \cdot x$

In PyTorch, a standard linear layer is represented by the formula:

$$
y = xW^T + b
$$

Where:
* **$x$** is the input vector/matrix.
* **$W$** contains the learnable weights.
* **$b$** is the learnable bias.
* **$y$** is the output.

In [9]:
import torch.nn as nn
layer = nn.Linear(in_features=1, out_features=1)

print(layer.weight)
print(layer.bias)

Parameter containing:
tensor([[-0.2804]], requires_grad=True)
Parameter containing:
tensor([0.1160], requires_grad=True)


nn.Linear creates trainable parameters for you. The initial values are generally randomized.
The arguments mean:
- in_features=1: each input example has one feature.
- out_features=1: the layer produces one output value per example.
For instance, if you were predicting house prices using only floor area, you might start with one input feature and one output prediction.
Important: nn.Linear is called a linear layer, although with its bias term it is technically an affine transformation. A neural network can learn nonlinear relationships by combining such layers with nonlinear activation functions.

### Inspecting input and output shapes

In [11]:
layer = nn.Linear(1,1)

x = torch.tensor([[1.0], [2.0], [3.0]])

y = layer(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)
print(y)

Input shape: torch.Size([3, 1])
Output shape: torch.Size([3, 1])
tensor([[-0.0528],
        [-0.9868],
        [-1.9207]], grad_fn=<AddmmBackward0>)


# Training my first nn.Linear model

In [12]:
import torch
import torch.nn as nn

# Training data: 3 examples, 1 feature each
X = torch.tensor([[1.0], [2.0], [3.0]])
y_true = torch.tensor([[2.0], [4.0], [6.0]])

# Model with one input and one output
model = nn.Linear(1, 1)

# Mean squared error
criterion = nn.MSELoss()

# Optimizer manages model parameters
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(500):
    # 1. Forward pass
    predictions = model(X)

    # 2. Calculate loss
    loss = criterion(predictions, y_true)

    # 3. Clear gradients
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Update parameters
    optimizer.step()

print("Learned weight:", model.weight.item())
print("Learned bias:", model.bias.item())
print("Final loss:", loss.item())

# Test predictions
with torch.no_grad():
    print("Predictions:")
    print(model(X))

Learned weight: 1.8968026638031006
Learned bias: 0.23459157347679138
Final loss: 0.007932940497994423
Predictions:
tensor([[2.1314],
        [4.0282],
        [5.9250]])
